# Data download and steps before using this notebook


- create data folder

- download all the data into the data folder

- see notes_about_data.txt to find out data that was used in this analysis

- TCIA files require NBIA data retriever and manually starting the data retrieval by double clicking on TCIA file (were not used in this study)

In [ ]:
# import libraries
import os
import sys
import zipfile
import tarfile
import re

In [ ]:
# directory to extract zip files
data_dir = 'data'
extract_dir = 'data'

In [ ]:
# define the key words in file names for each dataset
dataset_key_words = {
    'decathlon10': r'Task[0-1][0-9]',
    'kits19': 'kits19',
    'chaos': '3431873',
    'sliver': ['training-scans', 'training-labels', 'test-scans'],
}



In [ ]:
# find folders inside data
folders = os.listdir(data_dir)
extensions = set([i.split('.')[-1] for i in folders])
extensions_to_avoid = ['txt', 'pdf']
folders = [i for i in folders if i.split('.')[-1] not in extensions_to_avoid]
zip_folders = [i for i in folders if i.split('.')[-1]=='zip']
tart_folders = [i for i in folders if i.split('.')[-1]=='tar']

print(f"Zip folders included: {zip_folders}")
print(f"Tar.gz folders included: {tart_folders}")

In [ ]:
# create a dictionary list to store folder and filepaths
dataset_paths = []

In [ ]:
# extract zip files
for zip_folder in zip_folders:

    # find which dataset the zip file belongs to based on the key words in the file name: first it scans for key-value pairs where value is a string or regex, then it scans for key-value pairs where value is a list
    # since dataset_key_words has both string/regex and list values, we need to handle both cases
    dataset_name = [
        key for key,value in dataset_key_words.items()
        if any(re.search(v, zip_folder) for v in (value if isinstance(value,list) else [value]))
    ]

    print(f'\n\nProcessing zip file: {zip_folder} | Dataset identified as: {dataset_name}')
    
    # get the relative path to the zip file
    rel_zip_path = os.path.join(extract_dir, zip_folder)

    # create the folder name and path for the zip file extraction
    folder_to_create = zip_folder.replace('.zip','')
    folder_to_create_path = os.path.join(extract_dir, folder_to_create)

    with zipfile.ZipFile(rel_zip_path, 'r') as zip_ref:
        # zip_ref.extractall(folder_to_create_path)

        print(f'Extracted {zip_folder} to {folder_to_create_path}')


    # scan if there are any nested zip files and extract them too
    nested_folders = os.listdir(folder_to_create_path)

  
    for nested_folder in nested_folders:

        if nested_folder.endswith('zip'):

            # get the relative path to the nested zip file
            rel_nested_zip_path = os.path.join(folder_to_create_path, nested_folder)
            
            # create the folder name and path for nested zip file
            nested_folder_to_create = nested_folder.replace('.zip','')
            nested_folder_to_create_path = os.path.join(folder_to_create_path, nested_folder_to_create)
            print(f'Nested zip folder: {nested_folder}')

            with zipfile.ZipFile(rel_nested_zip_path, 'r') as nested_zip_ref:
                # nested_zip_ref.extractall(nested_folder_to_create_path)
                nested_zip_ref.close()

                # delete the nested zip file after extraction to save space
                os.remove(rel_nested_zip_path)

                print(f'Extracted {nested_folder} to {nested_folder_to_create_path}')

                dataset_paths.append({
                    'dataset_name': dataset_name[0] if dataset_name else 'unknown',
                    'dataset_path': nested_folder_to_create_path
                })

        
        else:
            print(f'No nested zip files found in {folder_to_create_path}')

            to_append = {
                'dataset_name': dataset_name[0] if dataset_name else 'unknown',
                'dataset_path': folder_to_create_path
            }

            if to_append not in  dataset_paths:

                dataset_paths.append(to_append)

In [ ]:
# extract tar files
for tar_folder in tart_folders:

    # obtain the dataset name using the keywords defined above
    dataset_name = [
        key for key, value in dataset_key_words.items()

        if any(re.search(v, tar_folder) for v in (value if isinstance(value, list) else [value]))
    ]

    print(f'\n\nProcessing tar file: {tar_folder} | Dataset identified as: {dataset_name}')

    # get the relative path to the tar file
    rel_tar_path = os.path.join(extract_dir, tar_folder)

    # create the folder name and path for the tar file extraction
    folder_to_create = tar_folder.replace('.tar','').replace('.gz','')
    folder_to_create_path = os.path.join(extract_dir, folder_to_create)

    with tarfile.open(rel_tar_path, 'r') as tar_ref:
        # tar_ref.extractall(folder_to_create_path)
        tar_ref.close()

        print(f'Extracted {tar_folder} to {folder_to_create_path}')


    # see if there are any nested tar files and extract them too
    nested_folders = os.listdir(folder_to_create_path)

    for nested_folder in nested_folders:
        if nested_folder.endswith('tar') or nested_folder.endswith('tar.gz'):

            # get the relative path to the nested tar file
            rel_nested_tar_path = os.path.join(folder_to_create_path, nested_folder)

            # create the folder name and path for nested tar file
            nested_folder_to_create = nested_folder.replace('.tar','').replace('.gz','')
            nested_folder_to_create_path = os.path.join(folder_to_create_path, nested_folder_to_create)
            print(f'Nested tar folder: {nested_folder}')

            with tarfile.open(rel_nested_tar_path, 'r') as nested_tar_ref:
                # nested_tar_ref.extractall(nested_folder_to_create_path)
                nested_tar_ref.close()

                # delete the nested tar file after extraction to save space
                os.remove(rel_nested_tar_path)

                print(f'Extracted {nested_folder} to {nested_folder_to_create_path}')


            dataset_paths.append({
                'dataset_name': dataset_name[0] if dataset_name else 'unknown',
                'dataset_path': nested_folder_to_create_path
            })

        else:
            print(f'No nested tar files found in {folder_to_create_path}')

            to_append  =  {
                'dataset_name': dataset_name[0] if dataset_name else 'unknown',
                'dataset_path': folder_to_create_path
            }

            if to_append not in dataset_paths:
                dataset_paths.append(to_append)

In [ ]:
dataset_paths

In [ ]:
dataset_files = []
extensions = []

In [ ]:
# get the files and their paths into a list

for dataset in dataset_paths:
    dataset_name = dataset['dataset_name']
    dataset_path = dataset['dataset_path']

    print(f'\n\nScanning files in dataset: {dataset_name} | Path: {dataset_path}')

    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            file_path = os.path.join(root, file)

            # print(f'Found file: {file_path} in dataset: {dataset_name}')
            print(f'Extension: {file.split('.')[-1]}')
            extensions.append(file.split('.')[-1])
            # dataset_files.append({
            #     'dataset_name': dataset_name,
            #     'file_path': file_path
            # })

In [ ]:
set(extensions)